# Figure A1 · Learning at matched experience (appendix)

One plot for every benchmark of the study: two rows — the collection return of the sampled training actors, smoothed over 25 epochs, above the greedy development primary metric at the saved labels — by one column per environment (Dark Key-to-Door, CountRecall, MazeRunner and the passive T-Maze), against measured charged training calls. The pre-declared thresholds are drawn (20 doors; 0.9 exact accuracy; 0.5 goal fraction, descriptive; T-Maze success 1.0 sustained through the endpoint, 0.95 as companion) and the charged calls to each seed's first crossing are tabulated; a T-Maze seed that solves the task and later falls back has no sustained crossing and is censored.

Both rows are drawn as curves without markers, and both are smoothed for drawing only: the collection return over 25 epochs and the development series over 3 saved labels, both centred with edge padding. The band is the min-max range of the three seeds. Every number that is read — the first crossings, the saved table — uses the raw series, and `development_smoothing = 1` draws them raw.

Only fits whose measured budget is the full 8M and whose development series reached the endpoint enter. A cell still training or still being scored is named in the legend as excluded and its column is left alone; nothing about it is written on a chart.

Descriptive: the estimand was declared after a partial view of Memo's series; collection and validation actors differ in action selection and roster, so the two rows are not a generalisation gap, and a faster early rise is not an endpoint difference. The estimators and the plotting code are in this notebook, the shared plumbing in [figures_common.py](figures_common.py).

In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figure is written (default notebooks/outputs/figures);
                          its tables and notes go to the sibling data/ folder
"""
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import (  # noqa: E402
    default_layout,
    markdown_preview,
    select_cells,
)

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures to  {FIGURES_DIR}\ntables to   {DATA_DIR}")

In [ ]:
import json
from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np
from figures_common import (
    ALL_ENVIRONMENTS,
    BAND_ALPHA,
    DRAWN_CELLS,
    ENVIRONMENTS,
    FULL_WIDTH,
    INK_FAINT,
    PANEL,
    SEEDS,
    BuiltFigure,
    CellStyle,
    composite_legend,
    fit_complete,
    line_handles,
    paper_style,
    pending_handles,
    standalone,
    style_axes,
)

THRESHOLDS = {
    "keydoor": ((20.0, "20 doors", "first"),),
    "countrecall": ((0.9, "0.9 accuracy", "first"),),
    "mazerunner": ((0.5, "0.5 goal fraction", "first"),),
    "tmaze": (
        (1.0, "1.0 sustained", "sustained"),
        (0.95, "0.95 sustained", "sustained"),
    ),
}
"""The declared learning-efficiency thresholds per environment, primary first
(study protocol, sections 7 and 11): charged calls to the first saved label at the
threshold on Key-to-Door, CountRecall and MazeRunner (half the goals on the development maps, a descriptive threshold); on the T-Maze the first saved label from which
greedy development success stays at the threshold through the endpoint
(Memo's own T-Maze metric), 0.95 as the companion."""


@dataclass(frozen=True)
class Figure4Options:
    seeds: Sequence[int] = SEEDS
    cells: Sequence[CellStyle] = DRAWN_CELLS
    environments: Sequence[str] = ALL_ENVIRONMENTS
    """Every benchmark of the study, one column each: the main text's three and the passive T-Maze."""
    smoothing_epochs: int = 25
    """Window of the moving average over the per-epoch collection return."""
    development_smoothing: int = 3
    """Labels in the centred moving average over the development series drawn
    (the curves are read for their shape, and a
    20-label series drawn raw reads as a zigzag). The saved table and every
    first crossing keep the raw values; 1 draws them raw."""
    budget_millions: float = 8.0
    name: str = "figureA1_learning_matched_experience"
    figsize: tuple[float, float] | None = None
    """``None`` sizes the composite from the number of columns."""
    panel_size: tuple[float, float] = PANEL


# ---------------------------------------------------------------- telemetry


def _records_in(line):
    """The JSON objects on one telemetry line. A line is normally one object; a
    fit resumed after a full disk can leave a truncated object with the next
    row appended to the same line, so every complete object on the line is
    kept and the truncated fragment is dropped."""
    try:
        return [json.loads(line)], 0
    except json.JSONDecodeError:
        pass
    decoder, found, position = json.JSONDecoder(), [], 0
    while (start := line.find("{", position)) >= 0:
        try:
            record, end = decoder.raw_decode(line, start)
        except json.JSONDecodeError:
            position = start + 1
            continue
        found.append(record)
        position = end
    return found, 1


def read_telemetry(run, repairs=None):
    """Every telemetry row of a run; ``repairs`` collects the lines that held a
    truncated fragment and how many complete rows were recovered from them."""
    path = run / "training_metrics.jsonl"
    rows = []
    for number, line in enumerate(path.read_text().splitlines(), 1):
        if not line.strip():
            continue
        records, damaged = _records_in(line)
        rows.extend(records)
        if damaged and repairs is not None:
            repairs.append(
                {"file": str(path), "line": number, "recovered": len(records)}
            )
    return rows


def collection_curve(rows):
    """The collection return of the sampled actors against measured charged calls."""
    rollout = sorted(
        (r for r in rows if r.get("panel") == "train-rollout"),
        key=lambda r: int(r["Epoch"]),
    )
    calls = np.asarray([int(r["total_frames"]) for r in rollout], dtype=np.int64)
    returns = np.asarray(
        [float(r["Average Total Return (Across All Env Names)"]) for r in rollout]
    )
    return calls, returns


def label_calls(rows):
    """Measured charged calls at each saved label. AMAGO sets ``epoch`` after logging,
    so the ``train-rollout`` row of telemetry epoch N-1 holds the counter of label N."""
    return {
        int(r["Epoch"]) + 1: int(r["total_frames"])
        for r in rows
        if r.get("panel") == "train-rollout"
    }


def development_series(run, calls):
    """(label, calls, primary) at every saved label with a measured budget."""
    record = json.loads((run / "development.json").read_text())
    points = [
        (int(r["epoch"]), calls[int(r["epoch"])], float(r["primary"]))
        for r in record["series"]
        if int(r["epoch"]) in calls
    ]
    return sorted(points)


def first_crossing(points, threshold):
    """Charged calls at the first label whose primary reaches ``threshold``."""
    return next((calls for _, calls, primary in points if primary >= threshold), None)


def sustained_crossing(points, threshold):
    """Charged calls at the first label from which the primary stays at or
    above ``threshold`` at every later saved label through the endpoint
    (``None`` when the endpoint itself is below it: a seed that solves the task
    and later falls back has no sustained crossing and is censored)."""
    for index, (_, calls, _) in enumerate(points):
        if all(primary >= threshold for _, _, primary in points[index:]):
            return calls
    return None


CROSSINGS = {"first": first_crossing, "sustained": sustained_crossing}


def moving_average(values, window):
    """Centred moving average with edge padding; identity when too short."""
    values = np.asarray(values, dtype=float)
    if window <= 1 or len(values) < window:
        return values
    padded = np.pad(values, (window // 2, window - 1 - window // 2), mode="edge")
    return np.convolve(padded, np.ones(window) / window, mode="valid")


# ------------------------------------------------------------------ compute


def compute_environment(environment, layout, options, sources):
    """Per cell: the seed mean and range of the smoothed collection return and of the
    development primary at the saved labels; the first-crossing rows; the excluded cells."""
    curves, crossings, excluded, repairs = {}, [], [], []
    for cell in options.cells:
        if not all(
            fit_complete(layout, environment, cell.condition, s) for s in options.seeds
        ):
            excluded.append(
                {
                    "environment": environment,
                    "condition": cell.condition,
                    "status": "incomplete or in-flight fit",
                }
            )
            continue
        collection, development = [], []
        for seed in options.seeds:
            run = layout.run_directory(environment, cell.condition, seed)
            rows = read_telemetry(run, repairs)
            calls, returns = collection_curve(rows)
            collection.append(
                (calls, moving_average(returns, options.smoothing_epochs))
            )
            points = development_series(run, label_calls(rows))
            development.append(points)
            for threshold, _, rule in THRESHOLDS[environment]:
                crossing = CROSSINGS[rule](points, threshold)
                crossings.append(
                    {
                        "environment": environment,
                        "condition": cell.condition,
                        "seed": seed,
                        "threshold": threshold,
                        "rule": rule,
                        "first_crossing_calls": crossing,
                        "censored": crossing is None,
                        "last_measured_calls": points[-1][1] if points else None,
                        "endpoint_primary": points[-1][2] if points else None,
                    }
                )
            sources.extend(
                layout.relative(run / n)
                for n in ("training_metrics.jsonl", "development.json")
            )
        length = min(len(calls) for calls, _ in collection)
        x = collection[0][0][:length] / 1e6
        ys = np.stack([returns[:length] for _, returns in collection])
        grid = sorted(
            set.intersection(*[{c for _, c, _ in pts} for pts in development])
        )
        table = np.stack(
            [
                [next(p for _, c, p in pts if c == g) for g in grid]
                for pts in development
            ]
        )
        smoothed = np.stack(
            [moving_average(row, options.development_smoothing) for row in table]
        )
        curves[cell.condition] = {
            "collection": (x, ys),
            "development": (np.asarray(grid) / 1e6, smoothed),
            "development_raw": (np.asarray(grid) / 1e6, table),
        }
    return {
        "environment": environment,
        "curves": curves,
        "crossings": crossings,
        "excluded": excluded,
        "repairs": repairs,
    }


# ------------------------------------------------------------------- drawing


def _series(ax, x, ys, cell):
    """The seed mean as a line over the min-max band of the seeds; no markers,
    so a long series reads as a curve."""
    ax.fill_between(
        x,
        ys.min(0),
        ys.max(0),
        color=cell.color,
        alpha=BAND_ALPHA,
        linewidth=0,
        zorder=cell.zorder,
    )
    ax.plot(x, ys.mean(0), **cell.line())


CALLS_LABEL = "Charged training calls (millions)"


def _calls_axis(ax, options, *, xlabel=True):
    """The shared x axis; in the composite one figure-level label serves every
    column, so the panels keep their ticks and drop the repeated text."""
    ax.set_xlim(0, options.budget_millions)
    ax.set_xticks([0, 2, 4, 6, 8])
    if xlabel:
        ax.set_xlabel(CALLS_LABEL)
    style_axes(ax)


def draw_collection_return(ax, data, options, *, ylabel=True, xlabel=True):
    spec = ENVIRONMENTS[data["environment"]]
    for cell in options.cells:
        if cell.condition in data["curves"]:
            _series(ax, *data["curves"][cell.condition]["collection"], cell)
    _calls_axis(ax, options, xlabel=xlabel)
    ax.set_title(spec.title)
    if ylabel:
        ax.set_ylabel("Collection return (sampled)")
    if data["environment"] in ("countrecall", "tmaze"):
        ax.set_ylim(-1, 1)


def excluded_cells(data, options):
    """The cells of ``options`` this environment has no complete fit for, for
    the legend (never written on the chart)."""
    missing = {r["condition"] for r in data["excluded"]}
    return [c for c in options.cells if c.condition in missing]


def draw_development_primary(ax, data, options, *, title=True, xlabel=True):
    threshold, threshold_label, _ = THRESHOLDS[data["environment"]][0]
    for cell in options.cells:
        if cell.condition in data["curves"]:
            _series(ax, *data["curves"][cell.condition]["development"], cell)
    ax.axhline(
        threshold, color=INK_FAINT, linestyle=(0, (1, 2)), linewidth=0.8, zorder=0
    )
    # The threshold sits on its line at the left edge, over a light box: the
    # curves cross the panel, so no free corner survives every environment.
    ax.text(
        0.015,
        threshold,
        threshold_label,
        transform=ax.get_yaxis_transform(),
        color=INK_FAINT,
        fontsize=8.5,
        va="center",
        ha="left",
        zorder=30,
        bbox={"facecolor": "white", "alpha": 0.7, "edgecolor": "none", "pad": 0.8},
    )
    _calls_axis(ax, options, xlabel=xlabel)
    if title:
        ax.set_title(ENVIRONMENTS[data["environment"]].title)
    if data["environment"] == "countrecall":
        ax.set_ylim(0, 1.0)
        ax.set_ylabel("Exact accuracy (greedy, dev.)")
    elif data["environment"] == "mazerunner":
        ax.set_ylim(0, 1.0)
        ax.set_ylabel("Goal fraction (greedy, dev.)")
    elif data["environment"] == "tmaze":
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Greedy success (dev.)")
    else:
        ax.set_ylim(0, 40)
        ax.set_ylabel("Doors per task (greedy, dev.)")


# --------------------------------------------------------------------- build


def build_figure4(layout, options=None):
    """Figure 4 from the telemetry and development series under ``layout``; every
    panel also standalone."""
    options = options or Figure4Options()
    sources = []
    environments = list(options.environments)
    data = {
        env: compute_environment(env, layout, options, sources) for env in environments
    }
    excluded_by_environment = {
        ENVIRONMENTS[e].title: excluded_cells(data[e], options) for e in environments
    }
    handles = line_handles(options.cells) + pending_handles(
        excluded_by_environment, options.cells, reason="excluded (fit in flight)"
    )
    width = options.figsize or (
        FULL_WIDTH if len(environments) > 1 else FULL_WIDTH * 0.32,
        6.2,
    )
    with paper_style():
        # Two rows (collection return, development primary), one column per benchmark.
        fig = plt.figure(figsize=width, layout="constrained")
        axes = np.asarray(fig.subplots(2, len(environments))).reshape(
            2, len(environments)
        )
        for column, environment in enumerate(environments):
            # The environment names the column once, above; the rows are told
            # apart by their y labels and one shared x label serves both.
            draw_collection_return(
                axes[0][column],
                data[environment],
                options,
                ylabel=column == 0,
                xlabel=False,
            )
            draw_development_primary(
                axes[1][column], data[environment], options, title=False, xlabel=False
            )
        fig.supxlabel(CALLS_LABEL, fontsize=8)
        composite_legend(fig, handles, ncol=4)
    size = options.panel_size
    panels = {}
    for environment in environments:
        drawn = line_handles(
            c for c in options.cells if c.condition in data[environment]["curves"]
        )
        panels[f"{environment}_collection_return"] = standalone(
            lambda ax, e=environment: draw_collection_return(ax, data[e], options),
            size=size,
            handles=drawn,
        )
        panels[f"{environment}_development_primary"] = standalone(
            lambda ax, e=environment: draw_development_primary(ax, data[e], options),
            size=size,
            handles=drawn,
        )
    crossings = [r for env in data.values() for r in env["crossings"]]
    # The drawn curves (seed mean and seed range, smoothed for drawing), so
    # the print-size figure redraws exactly what this figure draws.
    curve_rows = [
        {
            "environment": environment,
            "condition": condition,
            "series": series,
            "calls_millions": float(x[index]),
            "mean": float(ys[:, index].mean()),
            "low": float(ys[:, index].min()),
            "high": float(ys[:, index].max()),
        }
        for environment, found in data.items()
        for condition, curves in found["curves"].items()
        for series in ("collection", "development")
        for x, ys in [curves[series]]
        for index in range(len(x))
    ]
    excluded = [r for env in data.values() for r in env["excluded"]]
    notes = {
        "seeds": list(options.seeds),
        "environments": environments,
        "smoothing_epochs": options.smoothing_epochs,
        "development_smoothing_labels": options.development_smoothing,
        "smoothing": "drawing only: the collection return over "
        f"{options.smoothing_epochs} epochs and the development series over "
        f"{options.development_smoothing} saved labels, both centred with edge "
        "padding; the crossings and the saved tables use the raw series",
        "thresholds": {
            k: [{"threshold": t, "rule": r} for t, _, r in v]
            for k, v in THRESHOLDS.items()
            if k in environments
        },
        "present": {env: sorted(data[env]["curves"]) for env in environments},
        "excluded": excluded,
        "excluded_in_legend": {
            k: [c.condition for c in v] for k, v in excluded_by_environment.items()
        },
        "telemetry_repairs": [r for env in data.values() for r in env["repairs"]],
        "lower_row": "greedy development primary at the saved labels (64-task development roster)",
    }
    return BuiltFigure(
        options.name,
        fig,
        {"first_crossing": crossings, "excluded": excluded, "curves": curve_rows},
        notes,
        sources,
        panels,
    )

In [ ]:
# One appendix plot for every environment on every cell: the RSM family, the memoryless control and the four comparators, Memo
# with fixed segments among them; two rows, the collection return of the
# training actors above and the greedy development primary below.
options = Figure4Options(smoothing_epochs=25, cells=select_cells())
figure_learning = build_figure4(layout, options)
display(figure_learning.figure)

In [ ]:
# Charged calls to the first crossing of the declared thresholds, per seed (the
# T-Maze rows use the sustained rule), and which fits were left out.
display(Markdown(markdown_preview(figure_learning.tables["first_crossing"], limit=60)))
print("present: ", figure_learning.notes["present"])
print("excluded:", figure_learning.notes["excluded_in_legend"])

In [ ]:
# Save: the composite (PDF + PNG), every panel (PNG), the crossing table and
# the notes.
for path in figure_learning.save(FIGURES_DIR, DATA_DIR):
    print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)
figure_learning.close()